# ASL Neural Net: Train & Export Model

Train the ASL classifier and download the saved model for deployment.

In [ ]:
# Setup GPU
!pip install cupy-cuda12x -q

import numpy as np
import pickle
import os

try:
    import cupy as cp
    xp = cp
    GPU = True
    print(f"GPU: {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")
except:
    xp = np
    GPU = False
    print("Using CPU")

def to_numpy(arr):
    return arr.get() if GPU and hasattr(arr, 'get') else arr

In [ ]:
# Upload dataset
from google.colab import files
import pandas as pd

print("Upload sign_mnist_train.csv and sign_mnist_test.csv")
uploaded = files.upload()

In [ ]:
# Load and preprocess data
train_df = pd.read_csv('sign_mnist_train.csv')
test_df = pd.read_csv('sign_mnist_test.csv')

train_x = train_df.drop('label', axis=1).values.T.astype(np.float32)
train_y_raw = train_df['label'].values
test_x = test_df.drop('label', axis=1).values.T.astype(np.float32)
test_y_raw = test_df['label'].values

# Remove J (9) and Z (25)
train_mask = (train_y_raw != 9) & (train_y_raw != 25)
train_x, train_y_raw = train_x[:, train_mask], train_y_raw[train_mask]
test_mask = (test_y_raw != 9) & (test_y_raw != 25)
test_x, test_y_raw = test_x[:, test_mask], test_y_raw[test_mask]

# Remap labels
train_y_raw[train_y_raw > 9] -= 1
test_y_raw[test_y_raw > 9] -= 1
train_y = train_y_raw.reshape(1, -1)
test_y = test_y_raw.reshape(1, -1)

# Per-image normalization
train_x = (train_x - np.mean(train_x, axis=0, keepdims=True)) / (np.std(train_x, axis=0, keepdims=True) + 1e-8)
test_x = (test_x - np.mean(test_x, axis=0, keepdims=True)) / (np.std(test_x, axis=0, keepdims=True) + 1e-8)

# Convert to GPU
train_x, train_y = xp.asarray(train_x), xp.asarray(train_y)
test_x, test_y = xp.asarray(test_x), xp.asarray(test_y)

print(f"Train: {train_x.shape[1]} samples, Test: {test_x.shape[1]} samples")

In [ ]:
# Neural network functions
def relu(Z): return xp.maximum(0, Z), Z
def softmax(Z):
    e = xp.exp(Z - xp.max(Z, axis=0, keepdims=True))
    return e / xp.sum(e, axis=0, keepdims=True), Z

def init_params(dims):
    xp.random.seed(1)
    params = {}
    for l in range(1, len(dims)):
        params[f'W{l}'] = xp.random.randn(dims[l], dims[l-1]).astype(xp.float32) / xp.sqrt(dims[l-1])
        params[f'b{l}'] = xp.zeros((dims[l], 1), dtype=xp.float32)
    return params

def train_model(X, Y, layer_dims, lr=0.03, iterations=3000):
    params = init_params(layer_dims)
    costs = []
    L = len(layer_dims) - 1
    
    for i in range(iterations):
        # Forward
        A = X
        caches = [(X, None)]
        
        for l in range(1, L):
            Z = params[f'W{l}'].dot(A) + params[f'b{l}']
            A, _ = relu(Z)
            caches.append((A, Z))
        
        Z = params[f'W{L}'].dot(A) + params[f'b{L}']
        AL, _ = softmax(Z)
        
        # Cost
        m = Y.shape[1]
        Y_oh = xp.zeros_like(AL)
        Y_oh[Y.astype(int), xp.arange(m)] = 1
        cost = float(-xp.sum(Y_oh * xp.log(AL + 1e-8)) / m)
        
        # Backward
        dZ = AL - Y_oh
        for l in reversed(range(1, L + 1)):
            A_prev = caches[l-1][0]
            dW = (1/m) * xp.dot(dZ, A_prev.T)
            db = (1/m) * xp.sum(dZ, axis=1, keepdims=True)
            
            if l > 1:
                dA = xp.dot(params[f'W{l}'].T, dZ)
                dZ = dA * (caches[l-1][1] > 0)
            
            params[f'W{l}'] -= lr * dW
            params[f'b{l}'] -= lr * db
        
        if i % 100 == 0:
            costs.append(cost)
        if i % 500 == 0:
            print(f"Iteration {i}: Cost = {cost:.6f}")
    
    return params, costs

def predict(X, params, layer_dims):
    A = X
    L = len(layer_dims) - 1
    for l in range(1, L):
        Z = params[f'W{l}'].dot(A) + params[f'b{l}']
        A, _ = relu(Z)
    Z = params[f'W{L}'].dot(A) + params[f'b{L}']
    AL, _ = softmax(Z)
    return xp.argmax(AL, axis=0).reshape(1, -1)

In [ ]:
# Train
import time

layer_dims = [784, 128, 64, 24]
print(f"Architecture: {layer_dims}")
print(f"Training...\n")

start = time.time()
params, costs = train_model(train_x, train_y, layer_dims, lr=0.03, iterations=3000)
print(f"\nCompleted in {time.time() - start:.2f}s")

# Evaluate
train_pred = predict(train_x, params, layer_dims)
test_pred = predict(test_x, params, layer_dims)
train_acc = float(xp.mean(train_pred == train_y))
test_acc = float(xp.mean(test_pred == test_y))

print(f"\nTrain Accuracy: {train_acc:.2%}")
print(f"Test Accuracy:  {test_acc:.2%}")

In [ ]:
# Save model
model_data = {
    'layer_dims': layer_dims,
    'parameters': {k: to_numpy(v) for k, v in params.items()},
    'train_accuracy': train_acc,
    'test_accuracy': test_acc
}

with open('asl_neural_net_model.pkl', 'wb') as f:
    pickle.dump(model_data, f)

print("Model saved to asl_neural_net_model.pkl")
print(f"File size: {os.path.getsize('asl_neural_net_model.pkl') / 1024:.1f} KB")

# Download
files.download('asl_neural_net_model.pkl')

## Usage in Deployment

```python
import pickle
import numpy as np

# Load model
with open('asl_neural_net_model.pkl', 'rb') as f:
    model = pickle.load(f)

params = model['parameters']
layer_dims = model['layer_dims']

def predict(image):
    # image: 784 array (28x28 flattened)
    A = image.reshape(784, 1).astype(np.float32)
    A = (A - A.mean()) / (A.std() + 1e-8)  # Normalize
    
    L = len(layer_dims) - 1
    for l in range(1, L):
        Z = params[f'W{l}'].dot(A) + params[f'b{l}']
        A = np.maximum(0, Z)  # ReLU
    
    Z = params[f'W{L}'].dot(A) + params[f'b{L}']
    exp_Z = np.exp(Z - np.max(Z))
    probs = exp_Z / np.sum(exp_Z)
    
    pred_idx = int(np.argmax(probs))
    letters = 'ABCDEFGHIKLMNOPQRSTUVWXY'  # No J, Z
    return letters[pred_idx], float(probs[pred_idx])
```